# Task 3: RAG Core Logic and Evaluation Retrieval-Augmented Generation pipeline for CrediTrust complaint analysis. 
LLM: `Qwen/Qwen2.5-7B-Instruct` via HuggingFace Inference API (free, no GPU needed).

In [2]:
import sys
sys.path.insert(0, "..")

import os
import numpy as np
import pandas as pd
from dotenv import load_dotenv

from src.retriever import load_vector_store, build_retriever, VALID_CATEGORIES
from src.generator import build_llm_client, generate_answer, MODEL
from src.embedding import load_embedding_model, EMBEDDING_MODEL_NAME

load_dotenv("../.env")
print("HF_TOKEN set:", bool(os.getenv("HF_TOKEN")))

HF_TOKEN set: True


## 1. Load the Vector Store
Points to the Task 2 sample index (vector_store/) by default. For the full-scale pre-built store from the challenge dataset resources update VECTOR_STORE_DIR to point at that directory instead.

In [3]:
VECTOR_STORE_DIR = "../vector_store"
COLLECTION_NAME  = "complaints_sample"   # Task 2 collection name
# If using the pre-built full-scale store, update COLLECTION_NAME = "complaints"
 
collection = load_vector_store(VECTOR_STORE_DIR, COLLECTION_NAME)

Loaded collection 'complaints_sample' with 23,114 chunks


## 2. Load the Embedding Model
Must be the same model used to build the index (all-MiniLM-L6-v2). Using a different model here would make query vectors incomparable to the stored chunk vectors, producing meaningless retrieval results.

In [4]:
print(f"Loading embedding model: {EMBEDDING_MODEL_NAME}")
embed_model = load_embedding_model(EMBEDDING_MODEL_NAME)
print(f"Embedding dimension: {embed_model.get_sentence_embedding_dimension()}")

Loading embedding model: sentence-transformers/all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6567.65it/s]


Embedding dimension: 384


C:\Users\AAM3\AppData\Local\Temp\ipykernel_13120\1221997280.py:3: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Embedding dimension: {embed_model.get_sentence_embedding_dimension()}")


## 3. Build the Retriever

In [5]:
retrieve = build_retriever(collection, model=embed_model)

# Unfiltered search
print("Unfiltered: 'unauthorized charges'")
for r in retrieve("unauthorized charges", top_k=3):
    print(f"  [{r['similarity_score']:.3f}] {r['product_category']:<20} | complaint {r['complaint_id']}")

print()

# Filtered search -- Credit Card only
print("Filtered to Credit Card: 'unauthorized charges'")
for r in retrieve("unauthorized charges", top_k=3, product_category="Credit Card"):
    print(f"  [{r['similarity_score']:.3f}] {r['product_category']:<20} | complaint {r['complaint_id']}")

Unfiltered: 'unauthorized charges'
  [0.313] Credit Card          | complaint 3644732
  [0.282] Credit Card          | complaint 3809499
  [0.266] Savings Account      | complaint 6175934

Filtered to Credit Card: 'unauthorized charges'
  [0.313] Credit Card          | complaint 3644732
  [0.303] Credit Card          | complaint 2927411
  [0.282] Credit Card          | complaint 3809499


## 4. Connect to the LLM
Uses HuggingFace Inference API -- the model runs on HF's servers, not your machine. Requires a free HF account and token in .env.

In [6]:
client = build_llm_client()
 
# Test the connection
test_response = client.chat_completion(
    messages=[{"role": "user", "content": "In one sentence, what is RAG?"}],
    max_tokens=80,
)
print("LLM connection OK:", test_response.choices[0].message.content.strip())

LLM connection OK: RAG stands for Retrieval-Augmented Generation, a technique that combines retrieval-based methods with generative models to produce more accurate and contextually relevant responses.


## 5. Full RAG Pipeline
Wraps retrieve + generate into a single function for convenience.

In [7]:
def rag_pipeline(question: str, top_k: int = 5, product_category: str = None) -> dict:
    """
    End-to-end RAG:
      RETRIEVE  -> embed question, find top_k relevant chunks
                   (optionally filtered by product_category)
      AUGMENT   -> format chunks into context block + inject into prompt
      GENERATE  -> call LLM, return grounded answer with sources
    """
    retrieved = retrieve(question, top_k=top_k, product_category=product_category)
    result    = generate_answer(client, question, retrieved)
    return result


def show_result(result: dict, show_prompt: bool = False):
    print("=" * 72)
    print(f"QUESTION: {result['question']}")
    print("=" * 72)
    print(f"\nANSWER:\n{result['answer']}")
    print("\nSOURCES RETRIEVED:")
    for s in result["sources"]:
        print(f"  [{s['similarity_score']:.3f}] {s['product_category']:<20} | complaint {s['complaint_id']}")
    if show_prompt:
        print("\nFULL PROMPT SENT TO LLM:")
        print("-" * 40)
        print(result["prompt"])
    print("=" * 72)
    print()

## 6. Evaluation Queries
15 questions covering all 4 product categories, cross-product trends,filtered retrieval, and an out-of-scope fallback test.

In [8]:
# Credit Card
r1 = rag_pipeline("Why are customers unhappy with their credit cards?")
show_result(r1)

r2 = rag_pipeline(
    "What fraud or unauthorized charge issues are credit card customers reporting?",
    product_category="Credit Card",
)
show_result(r2)

r3 = rag_pipeline(
    "How are credit card companies handling billing disputes?",
    product_category="Credit Card",
)
show_result(r3)

# Savings Account
r4 = rag_pipeline("What problems are customers experiencing with their savings accounts?")
show_result(r4)

r5 = rag_pipeline(
    "Why are customers unable to access their savings accounts?",
    product_category="Savings Account",
)
show_result(r5)

r6 = rag_pipeline(
    "What unexpected fees are savings account customers being charged?",
    product_category="Savings Account",
)
show_result(r6)

# Money Transfer
r7 = rag_pipeline("What issues are customers facing with money transfers?")
show_result(r7)

r8 = rag_pipeline(
    "How long are customers waiting for their money transfers to complete?",
    product_category="Money Transfer",
)
show_result(r8)

r9 = rag_pipeline(
    "What happens when a money transfer fails or is lost?",
    product_category="Money Transfer",
)
show_result(r9)

# Personal Loan
r10 = rag_pipeline("What are the main complaints about personal loans?")
show_result(r10)

r11 = rag_pipeline(
    "How are loan servicers treating customers who are struggling to make payments?",
    product_category="Personal Loan",
)
show_result(r11)

# Cross-product
r12 = rag_pipeline("How are customers describing their experience with customer service?")
show_result(r12)

r13 = rag_pipeline("Which product category has the most complaints about hidden fees?")
show_result(r13)

r14 = rag_pipeline("What patterns appear across complaints from multiple product categories?")
show_result(r14)

# Out of scope -- fallback test
r15 = rag_pipeline("What is the current Bitcoin price?")
show_result(r15)
print("Fallback worked if answer says it does not have enough information.")

QUESTION: Why are customers unhappy with their credit cards?

ANSWER:
Customers are unhappy with their credit cards due to several reasons including:
- Sudden decreases in credit limits without prior notice.
- Lack of communication and confirmation of payments.
- Inaccurate or outdated credit scoring systems leading to higher interest rates.
- Not receiving credit card bills, making it difficult to manage payments.
- Security issues with the delivery of credit cards, leading to unauthorized use.

SOURCES RETRIEVED:
  [0.233] Credit Card          | complaint 4847774
  [0.174] Credit Card          | complaint 7568423
  [0.101] Credit Card          | complaint 7304692
  [0.099] Credit Card          | complaint 11827687
  [0.080] Credit Card          | complaint 12080058

QUESTION: What fraud or unauthorized charge issues are credit card customers reporting?

ANSWER:
Credit card customers are reporting issues related to fraudulent transactions and unauthorized charges. Specifically, they a

## 7. Embedding-Based Evaluation Metrics
Following the demo's approach: answer relevancy, context precision, and approximate faithfulness -- all computed locally with cosine similarity, zero extra API calls.

In [9]:
def eval_answer_relevancy(question: str, answer: str) -> float:
    """Cosine similarity between question and answer embeddings."""
    q_vec = embed_model.encode(question, normalize_embeddings=True)
    a_vec = embed_model.encode(answer,   normalize_embeddings=True)
    return float(np.dot(q_vec, a_vec))
 
 
def eval_context_precision(question: str, chunks: list) -> float:
    """Mean cosine similarity between question and each retrieved chunk."""
    q_vec = embed_model.encode(question, normalize_embeddings=True)
    sims  = []
    for chunk in chunks:
        c_vec = embed_model.encode(chunk["chunk_text"], normalize_embeddings=True)
        sims.append(float(np.dot(q_vec, c_vec)))
    return float(np.mean(sims)) if sims else 0.0
 
 
def eval_faithfulness_approx(answer: str, chunks: list) -> float:
    """Approx faithfulness: cosine sim between answer and combined context."""
    combined = " ".join(c["chunk_text"] for c in chunks)[:2000]
    a_vec = embed_model.encode(answer,   normalize_embeddings=True)
    c_vec = embed_model.encode(combined, normalize_embeddings=True)
    return float(np.dot(a_vec, c_vec))

In [12]:
eval_results = [r1, r2, r3, r4, r5, r6, r7, r8, r9, r10, r11, r12, r13, r14, r15]
eval_labels  = [
    "Why unhappy with credit cards?",
    "Fraud/unauthorized charges? [filtered: CC]",
    "Billing dispute handling? [filtered: CC]",
    "Problems with savings accounts?",
    "Unable to access savings accounts? [filtered: SA]",
    "Unexpected savings account fees? [filtered: SA]",
    "Issues with money transfers?",
    "Transfer completion times? [filtered: MT]",
    "Failed/lost transfers? [filtered: MT]",
    "Complaints about personal loans?",
    "Struggling payment treatment? [filtered: PL]",
    "Customer service experience?",
    "Which product has most hidden fee complaints?",
    "Patterns across product categories?",
    "Bitcoin price? (out of scope)",
]

## 8. Qualitative Evaluation Table (for report)

In [13]:
print("| # | Question | Generated Answer (summary) | Retrieved Sources | Quality Score (1-5) | Comments |")
print("|---|---|---|---|---|---|")
for i, (label, result) in enumerate(zip(eval_labels, eval_results), 1):
    answer_summary = result["answer"][:120].replace("\n", " ") + "..."
    sources = "; ".join(
        f"{s['product_category']} #{s['complaint_id']}"
        for s in result["sources"][:2]
    )
    print(f"| {i} | {label} | {answer_summary} | {sources} | [FILL SCORE] | [FILL COMMENT] |")

| # | Question | Generated Answer (summary) | Retrieved Sources | Quality Score (1-5) | Comments |
|---|---|---|---|---|---|
| 1 | Why unhappy with credit cards? | Customers are unhappy with their credit cards due to several reasons including: - Sudden decreases in credit limits with... | Credit Card #4847774; Credit Card #7568423 | [FILL SCORE] | [FILL COMMENT] |
| 2 | Fraud/unauthorized charges? [filtered: CC] | Credit card customers are reporting issues related to fraudulent transactions and unauthorized charges. Specifically, th... | Credit Card #6696016; Credit Card #2846172 | [FILL SCORE] | [FILL COMMENT] |
| 3 | Billing dispute handling? [filtered: CC] | Credit card companies are handling billing disputes in various ways based on the provided excerpts. Some customers repor... | Credit Card #6906046; Credit Card #2738462 | [FILL SCORE] | [FILL COMMENT] |
| 4 | Problems with savings accounts? | Customers are experiencing several problems with their savings accounts, including: - U

| # | Question | Generated Answer (summary) | Retrieved Sources | Quality Score (1-5) | Comments |
|---|---|---|---|---|---|
| 1 | Why are customers unhappy with their credit cards? | Sudden credit limit decreases, missed bills, inaccurate credit scoring, payment confirmation failures, and card delivery security issues. | Credit Card #4847774; Credit Card #7568423 | 4 | Good broad overview. All 5 sources are Credit Card complaints. Lower similarity scores (0.08–0.23) on some chunks suggest the question is wide enough to pull loosely related chunks, but the answer quality is solid. |
| 2 | What fraud or unauthorized charges are credit card customers reporting? [filtered] | Arbitrary charges without knowledge, overcharges, difficulty disputing, charges persisting after identity theft reports, and credit limits being exceeded. | Credit Card #6696016; Credit Card #2846172 | 5 | Best result. Filtering to Credit Card eliminated cross-category leakage seen in the unfiltered run. High similarity scores (0.35–0.41), specific and detailed answer, all claims grounded in sources. |
| 3 | How are credit card companies handling billing disputes? [filtered] | Companies vary widely — some withhold payments during disputes, others insinuate or lie about charges, fail to resolve effectively, or provide unsatisfactory resolutions. | Credit Card #6906046; Credit Card #2738462 | 4 | Strong result. Filter kept all sources within Credit Card. High similarity scores (0.31–0.35). Answer correctly reflects frustration rather than overstating resolution quality. |
| 4 | What problems are customers experiencing with their savings accounts? | Unexpected account freezes/holds, overdraft and service fees during hardship, payment processing delays, and fraudulent fund removal. | Savings Account #2829229; Savings Account #5285223 | 3 | Decent answer but one Credit Card chunk (#6525312) leaked into top-5 — unfiltered query. Low similarity scores overall (0.05–0.08) indicate the question is broad. Would improve with filter or full-scale store. |
| 5 | Why are customers unable to access their savings accounts? [filtered] | Technical errors, locked accounts with no callback, third-party bank blocks, and online login errors with unhelpful automated support. | Savings Account #4774883; Savings Account #5790869 | 5 | Excellent. Filtering eliminated the Money Transfer leakage seen in the original run. All 5 sources are Savings Account, similarity scores reasonable (0.07–0.12), answer is specific and well-structured. |
| 6 | What unexpected fees are savings account customers being charged? [filtered] | Account fees, transfer fees, anonymous draft fees, overdraft fees, and NSF fees — some customers facing multiple fees even when deposits cover purchases. | Savings Account #6751975; Savings Account #7263942 | 4 | Good targeted result. All sources are Savings Account. Strong similarity scores (0.18–0.31). LLM appended the fallback phrase despite giving a complete answer — minor prompt-following inconsistency, does not affect answer quality. |
| 7 | What issues are customers facing with money transfers? | Transaction restrictions, fraud concerns, refund disputes, transfer delays up to 2 months, and transfer process failures. | Money Transfer #7586955; Money Transfer #11638751 | 3 | Good answer but one Savings Account chunk leaked into top-5 (unfiltered). Similarity scores moderate (0.14–0.18). Adding a filter would have kept all sources on-product. |
| 8 | How long are customers waiting for money transfers to complete? [filtered] | LLM triggered fallback — stated it did not have enough specific timing information in the excerpts. | Money Transfer #3846481; Money Transfer #12118153 | 3 | Honest and correct fallback — the retrieved chunks discuss transfer problems but not specific durations. All sources are Money Transfer (filter worked). Low similarity scores (0.01–0.14) confirm the specific timing angle is not well-covered in the 12.5K sample. Would likely answer better with the full-scale store. |
| 9 | What happens when a money transfer fails or is lost? [filtered] | Customers unable to withdraw funds, difficulty reaching support, transfers that never complete, unconfirmed receipts, and delays from account number changes. | Money Transfer #3627796; Money Transfer #2768263 | 4 | Solid answer with concrete scenarios. All sources Money Transfer (filter worked). LLM appended fallback phrase despite giving a thorough answer — same prompt inconsistency as Q6. |
| 10 | What are the main complaints about personal loans? | Abusive collection agents, predatory lending with high APR, unfair startup fees, payment hardship during pandemic, and tribal sovereign immunity concerns. | Personal Loan #11504205; Personal Loan #11816292 | 4 | Good coverage of diverse loan complaint types. Two sources leaked from other categories (Savings Account, Credit Card) — unfiltered, reflects the smaller Personal Loan sample size. Answer remained on-topic despite leakage. |
| 11 | How are loan servicers treating struggling borrowers? [filtered] | LLM triggered fallback — stated excerpts do not specifically detail servicer treatment of struggling customers. | Personal Loan #1532021; Personal Loan #4281766 | 3 | Appropriate fallback. All 5 sources are Personal Loan (filter worked). Low similarity scores (0.10–0.13) confirm this specific angle is underrepresented in the 12.5K sample. Personal Loan has the smallest slice (971 complaints), limiting coverage of niche questions. |
| 12 | How are customers describing their experience with customer service? | Representatives described as indifferent, focused on maximising charges, unprofessional, hard to reach, with long hold times and language barriers. | Savings Account #1661213; Credit Card #3462976 | 3 | Weakest in-scope result. Near-zero and negative similarity scores (−0.01 to 0.07) confirm the retriever struggled with this broad cross-cutting question. Answer is vague compared to product-specific questions. Improvement requires either a larger vector store or a more targeted rephrasing. |
| 13 | Which product has the most complaints about hidden fees? | LLM correctly triggered fallback — stated it cannot count complaints across categories from the excerpts alone. | Credit Card #6696016; Savings Account #13571167 | 4 | Correct and honest response. The question asks for an aggregate count which RAG cannot answer from 5 chunks alone — this is expected behaviour, not a failure. The fallback guardrail worked as designed. |
| 14 | What patterns appear across complaints from multiple product categories? | Misrepresentation of products, unresolved charge disputes, poor customer support, problematic refund processes, and widespread dissatisfaction shared on BBB and social media. | Credit Card #7828613; Credit Card #12204794 | 2 | Weakest overall result. Near-zero and negative similarity scores (−0.12 to 0.003) show the retriever could not find relevant cross-category evidence. The answer sounds plausible but is largely generic. Cross-category trend questions are not well-suited to chunk-level retrieval without a larger index or a summarisation layer. |
| 15 | What is the current Bitcoin price? (out of scope) | I don't have enough information to answer that. | Money Transfer #9612585; Money Transfer #5560885 | 5 | Fallback guardrail worked perfectly. All retrieved chunks have negative similarity scores, confirming the question is completely out of domain. The LLM correctly refused to hallucinate. |